# Validate Standardized Culture Organism

Converted from `sql/analysis/07_validate_standardized_culture_organism.sql`.

Validate standardized culture-organism reconciliation

Multiple source organism labels can map to the same
standardized organism within a single culture. Because ARMD
does not provide a verified isolate identifier, these cases
are flagged rather than arbitrarily collapsed.

1. Reconciliation outcomes by cohort.

In [1]:
%sql
select
    cohort_id,
    mapping_resolution_status,
    count(*) as standardized_culture_organism_rows
from workspace.micro_dev.int_standardized_culture_organism
group by
    cohort_id,
    mapping_resolution_status
order by
    cohort_id,
    mapping_resolution_status;

,cohort_id,mapping_resolution_status,standardized_culture_organism_rows
0,blood,collision,11
1,blood,resolved,9011
2,respiratory,collision,1721
3,respiratory,resolved,14048
4,urine,collision,58
5,urine,resolved,81866


2. Collision burden by standardized organism.

In [2]:
%sql
select
    cohort_id,
    organism_standardized,
    count(*) as collision_cultures
from workspace.micro_dev.int_standardized_culture_organism
where mapping_resolution_status = 'collision'
group by
    cohort_id,
    organism_standardized
order by
    collision_cultures desc,
    cohort_id,
    organism_standardized;

,cohort_id,organism_standardized,collision_cultures
0,respiratory,Pseudomonas aeruginosa,1520
1,respiratory,Staphylococcus aureus,200
2,urine,Pseudomonas aeruginosa,52
3,blood,Staphylococcus aureus,9
4,urine,Enterobacter cloacae complex,3
5,urine,Klebsiella pneumoniae,2
6,blood,Enterococcus faecium,1
7,blood,Escherichia coli,1
8,respiratory,Enterobacter cloacae complex,1
9,urine,Escherichia coli,1


3. Confirm resolved records contain exactly one source
culture-organism record.
Expected result: zero rows.

In [3]:
%sql
select
    standardized_culture_organism_key,
    cohort_id,
    organism_standardized,
    source_organism_record_count,
    source_labels
from workspace.micro_dev.int_standardized_culture_organism
where mapping_resolution_status = 'resolved'
  and (
      source_organism_record_count <> 1
      or culture_organism_key is null
  );

,standardized_culture_organism_key,cohort_id,organism_standardized,source_organism_record_count,source_labels


4. Confirm the standardized grain is unique.
Expected result: zero rows.

In [4]:
%sql
select
    cohort_id,
    culture_key,
    organism_standardized,
    count(*) as row_count
from workspace.micro_dev.int_standardized_culture_organism
group by
    cohort_id,
    culture_key,
    organism_standardized
having count(*) > 1;

,cohort_id,culture_key,organism_standardized,row_count


5. Quantify the analytical cohort remaining after ambiguous
standardized-organism collisions are excluded.

In [5]:
%sql
select
    cohort_id,
    organism_standardized,
    count(*) as resolved_culture_organism_rows,
    count(distinct patient_key) as patients
from workspace.micro_dev.int_standardized_culture_organism
where mapping_resolution_status = 'resolved'
group by
    cohort_id,
    organism_standardized
order by
    cohort_id,
    resolved_culture_organism_rows desc;

,cohort_id,organism_standardized,resolved_culture_organism_rows,patients
0,blood,Escherichia coli,3338,3021
1,blood,Staphylococcus aureus,1922,1731
2,blood,Klebsiella pneumoniae,1137,1050
3,blood,Enterococcus faecalis,568,530
4,blood,Pseudomonas aeruginosa,411,384
5,blood,Enterococcus faecium,290,268
6,blood,Streptococcus agalactiae,277,262
7,blood,Enterobacter cloacae complex,257,243
8,blood,Klebsiella oxytoca,227,212
9,blood,Proteus mirabilis,225,218
